The attached files are a collection of tweets labelled with sentiment in 3 categories:

sentiments = {
    "LABEL_0": "Bearish", 
    "LABEL_1": "Bullish", 
    "LABEL_2": "Neutral"
}

Train a LSTM network to with the training file. Validate the trained model with the valid file. Comment what you are doing in each part of your code. As the better the code, comments and result validation as the better the grade.

In [6]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Load the dataset, each tweet of the dataset have a prompt that is the tweet text and the label that is the type of the email (0,1,2)
file_path_train = "/Users/bernardoquindimil/Code/Berniquindimil/NLP_Digital_Portfolio/S09/sent_train.csv"
df_train = pd.read_csv(file_path_train)

file_path_valid = "/Users/bernardoquindimil/Code/Berniquindimil/NLP_Digital_Portfolio/S09/sent_valid.csv"
df_test = pd.read_csv(file_path_valid)

In [8]:
# The dataset has 'text' and 'label' columns
texts_train = df_train['text'].astype(str).values  # Convert to string in case of NaN
labels_train = df_train['label'].values

# Encode labels
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels_train)

## Tokenization

In [9]:
max_words = 10000  # Tamaño máximo del vocabulario
max_len = 200  # Longitud máxima de la secuencia
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts_train)
sequences = tokenizer.texts_to_sequences(texts_train)
X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)

## LSTM model

In [ ]:
# Create the LSTM model
model_lstm = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(64, return_sequences=True),
    Dropout(0.5),
    LSTM(32),
    Dense(32, activation='relu'),
    Dense(1, activation='softmax')  # Softmax for multiple class
])

/Users/bernardoquindimil/miniconda3/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


The RNNs has the disadvantage that store a lot of words for 

In [ ]:
model_lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])  # Binary cossentropy because in most of cases, is bettet than one. Binary crossentropy means that not only takes in account the order from left to right.

In [ ]:
# Training the LSTM model
model_lstm.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=32) # Train with the sent_train.csv data doing a partition and validate with the sent_valid.csv

Epoch 1/5


/Users/bernardoquindimil/miniconda3/lib/python3.11/site-packages/keras/src/ops/nn.py:827: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


239/239 ━━━━━━━━━━━━━━━━━━━━ 26s 104ms/step - accuracy: 0.1920 - loss: -7.1478 - val_accuracy: 0.2048 - val_loss: -39.4106
Epoch 2/5
239/239 ━━━━━━━━━━━━━━━━━━━━ 25s 106ms/step - accuracy: 0.2005 - loss: -58.3914 - val_accuracy: 0.2048 - val_loss: -132.4062
Epoch 3/5
239/239 ━━━━━━━━━━━━━━━━━━━━ 25s 104ms/step - accuracy: 0.1925 - loss: -168.4701 - val_accuracy: 0.2048 - val_loss: -280.0293
Epoch 4/5
239/239 ━━━━━━━━━━━━━━━━━━━━ 27s 113ms/step - accuracy: 0.1958 - loss: -327.0081 - val_accuracy: 0.2048 - val_loss: -478.8745
Epoch 5/5
239/239 ━━━━━━━━━━━━━━━━━━━━ 26s 110ms/step - accuracy: 0.2048 - loss: -535.7217 - val_accuracy: 0.2048 - val_loss: -724.1073


In [14]:
# Evaluate LSTM model
loss_lstm, accuracy_lstm = model_lstm.evaluate(X_test, y_test)
print(f"LSTM Test Accuracy: {accuracy_lstm:.4f}")

60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.2247 - loss: -688.7380
LSTM Test Accuracy: 0.2048


I have an accuracy of 0.2048 that is not good because I have done very few eppoch. A solution is training with more eppochs